In [1]:
import os
from pathlib import Path

HOME_DIR = os.path.dirname(os.getcwd())
DATA_DIR = Path("/data/dacon-236105")
MODEL_DIR = Path(HOME_DIR) / "model"
OUTPUT_DIR = Path(HOME_DIR) / "output"

MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# EDA 2. Baselines: Linear Mel / Mel + MFCC → BiLSTM

- Model 1: 256-bin linear (power) mel spectrogram → BiLSTM → 6 classes
- Model 2: 256-bin linear mel spectrogram ⊕ MFCC (frame-wise concat) → BiLSTM → 6 classes
- Stratified 5-fold, AdamW + ReduceLROnPlateau, fold 모델 softmax 평균으로 test 추론

## Import

In [2]:
import random
import time
import warnings

import librosa
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.io import wavfile
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings(action="ignore")

/home/hommy/Repositories/dacon-236105/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Hyperparameter Setting

In [3]:
CFG = {
    # Feature
    "SR": 16000,
    "N_FFT": 1024,       # n_mels=256일 때 n_fft=400/512는 empty mel filter 발생 → 1024
    "WIN_LENGTH": 400,   # 25 ms
    "HOP_LENGTH": 160,   # 10 ms
    "N_MELS": 256,
    "LOG_MEL": False,    # False: linear(power) mel, True: log-mel(dB)
    "N_MFCC": 40,        # MFCC는 log-mel에서 계산

    # Model
    "N_CLASSES": 6,
    "HIDDEN": 256,
    "NUM_LAYERS": 2,
    "DROPOUT": 0.3,

    # Training
    "N_FOLDS": 5,
    "BATCH_SIZE": 64,
    "EPOCHS": 500,
    "LR": 1e-4,
    "WEIGHT_DECAY": 1e-2,
    "LR_FACTOR": 0.5,    # ReduceLROnPlateau
    "LR_PATIENCE": 8,    # ReduceLROnPlateau
    "EARLY_STOP": 20,    # best val_acc 대비 20 epoch 연속 개선 없으면 중단 (저장은 best epoch만)
    "GRAD_CLIP": 5.0,
    "SEED": 42,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Fixed Random-Seed

In [4]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["SEED"]) # Seed 고정

## Data Pre-Processing

In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

train_df["path"] = train_df["path"].apply(lambda p: str(DATA_DIR / p))
test_df["path"] = test_df["path"].apply(lambda p: str(DATA_DIR / p))

labels = train_df["label"].values
len(train_df), len(test_df)

(5001, 1881)

### Feature Extraction

In [6]:
def load_wav(path):
    sr, y = wavfile.read(path)
    assert sr == CFG["SR"] and y.dtype == np.int16 and y.ndim == 1, (path, sr, y.dtype, y.shape)
    return y.astype(np.float32) / 32768.0

def get_features(path):
    """Returns (mel [T, N_MELS], mfcc [T, N_MFCC]) as float32."""
    y = load_wav(path)
    mel = librosa.feature.melspectrogram(
        y=y, sr=CFG["SR"], n_fft=CFG["N_FFT"], win_length=CFG["WIN_LENGTH"],
        hop_length=CFG["HOP_LENGTH"], n_mels=CFG["N_MELS"], power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=1.0)
    mfcc = librosa.feature.mfcc(S=log_mel, n_mfcc=CFG["N_MFCC"])
    mel_feat = log_mel if CFG["LOG_MEL"] else mel
    return mel_feat.T.astype(np.float32), mfcc.T.astype(np.float32)

def extract_features(df):
    mels, mfccs = [], []
    for path in tqdm(df["path"]):
        mel, mfcc = get_features(path)
        mels.append(mel)
        mfccs.append(mfcc)
    return mels, mfccs

In [7]:
train_mel, train_mfcc = extract_features(train_df)
test_mel, test_mfcc = extract_features(test_df)

train_mel_mfcc = [np.concatenate([m, c], axis=1) for m, c in zip(train_mel, train_mfcc)]
test_mel_mfcc = [np.concatenate([m, c], axis=1) for m, c in zip(test_mel, test_mfcc)]

train_mel[0].shape, train_mel_mfcc[0].shape

  0%|          | 0/5001 [00:00<?, ?it/s]

  0%|          | 1/5001 [00:00<37:26,  2.23it/s]

  0%|          | 7/5001 [00:00<05:16, 15.79it/s]

  0%|          | 13/5001 [00:00<03:07, 26.65it/s]

  1%|          | 28/5001 [00:00<01:24, 58.89it/s]

  1%|          | 45/5001 [00:00<00:55, 88.64it/s]

... (263 outputs left out)

 98%|█████████▊| 1852/1881 [00:07<00:00, 273.34it/s]

100%|█████████▉| 1880/1881 [00:07<00:00, 256.92it/s]

100%|██████████| 1881/1881 [00:07<00:00, 244.31it/s]

((237, 256), (237, 296))

### Normalization

In [8]:
def fit_normalizer(feats):
    n = sum(len(f) for f in feats)
    s = sum(f.sum(axis=0, dtype=np.float64) for f in feats)
    ss = sum((f.astype(np.float64) ** 2).sum(axis=0) for f in feats)
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

## Dataset / DataLoader

In [9]:
class AudioFeatDataset(Dataset):
    def __init__(self, feats, labels=None, mean=None, std=None):
        self.feats = feats
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] - self.mean) / self.std
        y = -1 if self.labels is None else int(self.labels[idx])
        return torch.from_numpy(x), y

def collate_fn(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    x_pad = pad_sequence(xs, batch_first=True)  # [B, T_max, D]
    return x_pad, lengths, torch.tensor(ys, dtype=torch.long)

## Model

In [10]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x, lengths):
        # x: [B, T, D]
        B, T, D = x.shape
        scores = self.score(x).squeeze(-1)  # [B, T]
        mask = (
            torch.arange(T, device=x.device)[None, :]
            < lengths.to(x.device)[:, None]
        )
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)  # [B, T]
        pooled = torch.sum(
            x * weights.unsqueeze(-1), dim=1
        )  # [B, D]
        return pooled

class BiLSTMClassifier(nn.Module):
    def __init__(self, in_dim, hidden, num_layers, n_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            in_dim, hidden, num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pool = AttentionPooling(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, n_classes))

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, out_lens = pad_packed_sequence(out, batch_first=True)  # [B, T_max, 2H]
        pooled = self.pool(out, out_lens)  # masked attention pooling
        return self.head(pooled)

## Train / Validation

In [11]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, probs, ys = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                optimizer.step()
            total_loss += loss.item() * len(y)
            probs.append(logits.softmax(dim=-1).detach().cpu())
            ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    return total_loss / len(ys), probs, ys

@torch.no_grad()
def predict(model, loader):
    model.eval()
    return np.concatenate([
        model(x.to(device), lengths).softmax(dim=-1).cpu().numpy() for x, lengths, _ in loader
    ])

def make_loader(feats, labels, mean, std, shuffle, seed=None):
    return DataLoader(
        AudioFeatDataset(feats, labels, mean, std),
        batch_size=CFG["BATCH_SIZE"], shuffle=shuffle, collate_fn=collate_fn,
        generator=torch.Generator().manual_seed(seed) if shuffle else None,
    )

In [12]:
def run_cv(name, train_feats, test_feats, labels):
    skf = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    oof = np.zeros((len(train_feats), CFG["N_CLASSES"]), dtype=np.float32)
    test_prob = np.zeros((len(test_feats), CFG["N_CLASSES"]), dtype=np.float32)
    fold_logs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        seed_everything(CFG["SEED"] + fold)
        tr_feats = [train_feats[i] for i in tr_idx]
        va_feats = [train_feats[i] for i in va_idx]
        mean, std = fit_normalizer(tr_feats)

        tr_loader = make_loader(tr_feats, labels[tr_idx], mean, std, shuffle=True, seed=CFG["SEED"] + fold)
        va_loader = make_loader(va_feats, labels[va_idx], mean, std, shuffle=False)
        te_loader = make_loader(test_feats, None, mean, std, shuffle=False)

        model = BiLSTMClassifier(
            train_feats[0].shape[1], CFG["HIDDEN"], CFG["NUM_LAYERS"], CFG["N_CLASSES"], CFG["DROPOUT"],
        ).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=CFG["LR_FACTOR"], patience=CFG["LR_PATIENCE"],
        )

        ckpt_path = MODEL_DIR / f"{name}_fold{fold}.pt"
        best_acc, best_epoch, wait = -1.0, 0, 0
        t0 = time.time()
        for epoch in range(1, CFG["EPOCHS"] + 1):
            tr_loss, _, _ = run_epoch(model, tr_loader, criterion, optimizer)
            va_loss, va_prob, va_y = run_epoch(model, va_loader, criterion)
            va_acc = accuracy_score(va_y, va_prob.argmax(axis=1))
            scheduler.step(va_acc)

            improved = va_acc > best_acc
            if improved:
                best_acc, best_epoch, wait = va_acc, epoch, 0
                torch.save({
                    "model": model.state_dict(),
                    "mean": torch.from_numpy(mean), "std": torch.from_numpy(std),
                    "in_dim": train_feats[0].shape[1], "epoch": epoch, "cfg": CFG,
                }, ckpt_path)
            else:
                wait += 1
            lr = optimizer.param_groups[0]["lr"]
            mark = " *" if improved else ''
            print(f"[{name} | fold {fold}] ep {epoch:02d} | tr_loss {tr_loss:.4f} | va_loss {va_loss:.4f} "
                  f"| va_acc {va_acc:.4f} | lr {lr:.1e}{mark}")
            if wait >= CFG["EARLY_STOP"]:
                break
        train_time = time.time() - t0

        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        oof[va_idx] = predict(model, va_loader)
        test_prob += predict(model, te_loader) / CFG["N_FOLDS"]

        va_pred = oof[va_idx].argmax(axis=1)
        fold_logs.append({
            "fold": fold, "best_epoch": best_epoch,
            "val_acc": accuracy_score(labels[va_idx], va_pred),
            "val_macro_f1": f1_score(labels[va_idx], va_pred, average="macro"),
            "train_time_sec": round(train_time, 1),
        })
        fold_acc = fold_logs[-1]["val_acc"]
        print(f"[{name} | fold {fold}] best ep {best_epoch} | val_acc {fold_acc:.4f}\n")

    fold_logs = pd.DataFrame(fold_logs)
    oof_pred = oof.argmax(axis=1)
    oof_acc = accuracy_score(labels, oof_pred)
    oof_f1 = f1_score(labels, oof_pred, average="macro")
    print(f"[{name}] OOF acc {oof_acc:.4f} | OOF macro-F1 {oof_f1:.4f}")
    return oof, test_prob, fold_logs

## Submission Helper

In [13]:
def save_submission(test_prob, fname):
    submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
    pred = pd.Series(test_prob.argmax(axis=1), index=test_df["id"].values)
    submission["label"] = submission["id"].map(pred)
    assert submission["label"].notna().all()
    submission["label"] = submission["label"].astype(int)
    submission.to_csv(OUTPUT_DIR / fname, index=False)
    return submission

## Model 1. Linear Mel (256) → BiLSTM

In [14]:
oof_mel, test_prob_mel, logs_mel = run_cv("baseline_mel", train_mel, test_mel, labels)
display(logs_mel)

[baseline_mel | fold 0] ep 01 | tr_loss 1.7138 | va_loss 1.5940 | va_acc 0.3087 | lr 1.0e-04 *


[baseline_mel | fold 0] ep 02 | tr_loss 1.5477 | va_loss 1.5226 | va_acc 0.3397 | lr 1.0e-04 *


[baseline_mel | fold 0] ep 03 | tr_loss 1.4945 | va_loss 1.4742 | va_acc 0.3936 | lr 1.0e-04 *


[baseline_mel | fold 0] ep 04 | tr_loss 1.4551 | va_loss 1.4740 | va_acc 0.3946 | lr 1.0e-04 *


[baseline_mel | fold 0] ep 05 | tr_loss 1.4342 | va_loss 1.4303 | va_acc 0.4066 | lr 1.0e-04 *


[baseline_mel | fold 0] ep 06 | tr_loss 1.4069 | va_loss 1.4249 | va_acc 0.4226 | lr 1.0e-04 *


... (281 outputs left out)

[baseline_mel | fold 4] ep 60 | tr_loss 0.6932 | va_loss 2.0255 | va_acc 0.4520 | lr 2.5e-05


[baseline_mel | fold 4] ep 61 | tr_loss 0.6896 | va_loss 2.0429 | va_acc 0.4600 | lr 2.5e-05


[baseline_mel | fold 4] best ep 41 | val_acc 0.4730

[baseline_mel] OOF acc 0.4697 | OOF macro-F1 0.4590


,fold,best_epoch,val_acc,val_macro_f1,train_time_sec
0,0,46,0.458541,0.449957,262.6
1,1,12,0.440000,0.409873,124.7
2,2,55,0.503000,0.497400,297.5
3,3,31,0.474000,0.464472,202.2
4,4,41,0.473000,0.464257,242.2


In [15]:
submission_mel = save_submission(test_prob_mel, "baseline_mel.csv")
display(submission_mel["label"].value_counts().sort_index())

label
0    316
1    198
2    407
3    282
4    377
5    301
Name: count, dtype: int64

## Model 2. Linear Mel (256) + MFCC → BiLSTM

In [16]:
oof_mel_mfcc, test_prob_mel_mfcc, logs_mel_mfcc = run_cv("baseline_mel_mfcc", train_mel_mfcc, test_mel_mfcc, labels)
display(logs_mel_mfcc)

[baseline_mel_mfcc | fold 0] ep 01 | tr_loss 1.6895 | va_loss 1.5389 | va_acc 0.3427 | lr 1.0e-04 *


[baseline_mel_mfcc | fold 0] ep 02 | tr_loss 1.4778 | va_loss 1.4374 | va_acc 0.4066 | lr 1.0e-04 *


[baseline_mel_mfcc | fold 0] ep 03 | tr_loss 1.3984 | va_loss 1.3707 | va_acc 0.4446 | lr 1.0e-04 *


[baseline_mel_mfcc | fold 0] ep 04 | tr_loss 1.3439 | va_loss 1.3446 | va_acc 0.4675 | lr 1.0e-04 *


[baseline_mel_mfcc | fold 0] ep 05 | tr_loss 1.2993 | va_loss 1.3156 | va_acc 0.4795 | lr 1.0e-04 *


[baseline_mel_mfcc | fold 0] ep 06 | tr_loss 1.2668 | va_loss 1.2994 | va_acc 0.4715 | lr 1.0e-04


... (206 outputs left out)

[baseline_mel_mfcc | fold 4] ep 35 | tr_loss 0.2637 | va_loss 1.9767 | va_acc 0.5210 | lr 2.5e-05


[baseline_mel_mfcc | fold 4] ep 36 | tr_loss 0.2533 | va_loss 1.9975 | va_acc 0.5260 | lr 2.5e-05


[baseline_mel_mfcc | fold 4] best ep 16 | val_acc 0.5350

[baseline_mel_mfcc] OOF acc 0.5417 | OOF macro-F1 0.5382


,fold,best_epoch,val_acc,val_macro_f1,train_time_sec
0,0,19,0.54046,0.535768,158.9
1,1,27,0.54800,0.545669,191.5
2,2,26,0.55000,0.546018,187.8
3,3,22,0.53500,0.534481,170.8
4,4,16,0.53500,0.529278,146.9


In [17]:
submission_mel_mfcc = save_submission(test_prob_mel_mfcc, "baseline_mel_mfcc.csv")
display(submission_mel_mfcc["label"].value_counts().sort_index())

label
0    321
1    265
2    370
3    260
4    306
5    359
Name: count, dtype: int64

## Comparison

In [18]:
pd.DataFrame([
    {
        "model": name,
        "oof_acc": accuracy_score(labels, oof.argmax(axis=1)),
        "oof_macro_f1": f1_score(labels, oof.argmax(axis=1), average="macro"),
        "fold_acc_mean": logs["val_acc"].mean(),
        "fold_acc_std": logs["val_acc"].std(),
    }
    for name, oof, logs in [
        ("baseline_mel", oof_mel, logs_mel),
        ("baseline_mel_mfcc", oof_mel_mfcc, logs_mel_mfcc),
    ]
])

,model,oof_acc,oof_macro_f1,fold_acc_mean,fold_acc_std
0,baseline_mel,0.469706,0.458963,0.469708,0.023156
1,baseline_mel_mfcc,0.541692,0.538234,0.541692,0.007069
